In [1]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [3]:
results

[{'id': '30fbb4f5b8',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Confluent Kafka: Where can I find schema registry URL?',
  'answer': 'In [Confluent Cloud](https://confluent.cloud/):\n\n- Navigate to your Environment (e.g., default or a custom name).\n- Use the right navigation bar to find "Stream Governance API."\n- The URL can be found under "Endpoint."\n- Create credentials from the Credentials section below it.'},
 {'id': '477104b863',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Python Kafka: ./spark-submit.sh streaming.py - ERROR StandaloneSchedulerBackend: Application has been killed. Reason: All masters are unresponsive! Giving up.',
  'answer': 'While following [tutorial 13.2](https://www.youtube.com/watch?v=5hRJ8-6Fpyk&list=PL3MmuxUbc_hJed7dXYoJw8DoCuVHhGEQb&index=79), when running `./spark-submit.sh streaming.py`, encountered the following error:\n\n```\n24/03/11 09:48:36 INFO Sta

In [4]:
from rag_helper import RAGBase

In [5]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [6]:
import os

In [7]:
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [9]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=client,
)

In [11]:
vector_assistant.rag("How do I run Kafka?")

"Unfortunately, the provided context does not contain information on how to run Kafka. I don't know how to run Kafka based on the given context."